# CONTEXTUAL EMBEDDING = TRUE
# NGRAM = TRUE
# MIN_COUNT = 10
# EMBEDDER = all-MiniLM-L6-v2
# new preprocesssing func

In [1]:
!pip install gensim top2vec


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from top2vec import Top2Vec
import pandas as pd
from bertopic import BERTopic
from bertopic.backend import MultiModalBackend
from bertopic.representation import KeyBERTInspired
import re
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import random
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

c:\Users\Allen\Documents\Python Env\environments\derp_learning\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv(r"C:\Users\Allen\Documents\GitHub\thesis-research\combined_tweets_dataset\sample_v2\vibe_coding_combined_translated.csv")

In [4]:
df = df[["full_text_translated", "image_url"]]

In [5]:
df.head()

,full_text_translated,image_url
0,@karpathy me vibe coding and hope it can just ...,https://pbs.twimg.com/media/Gi0a82HaQAA97Q-.jpg
1,vibes capital meets vibe coding = greatest eco...,NaN
2,vibe coding https://t.co/1hHVMmunrw,https://pbs.twimg.com/media/Gi0f9fAWgAAgox6.jpg
3,can attest. vibe coding is hella fun and borde...,NaN
4,I have built a few app ideas in my spare time ...,NaN


In [6]:
import html
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Pastikan sudah download resource nltk
# import nltk
# nltk.download('punkt')
# nltk.download('stopwords')

stop_words = set(stopwords.words('english')) # Atau 'indonesian' jika data campur

def preprocess_tweet_v2(text):
    # 1. Decode HTML Entities (Penting untuk X/Twitter data)
    # Ini akan merubah &lt; menjadi < sehingga re.sub simbol bisa bekerja
    text = html.unescape(text)

    text = text.lower()

    # 2. Pola Regex kamu sudah bagus, pertahankan untuk URL, Mentions, Hashtags
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)

    # 3. Handle spesifik Vibe Coding sebelum hapus simbol
    text = re.sub(r'vibe\s+coding', 'vibecoding', text)
    text = re.sub(r'vibe\s+code', 'vibecode', text)
    text = re.sub(r'vibe\s+coded', 'vibecoded', text)

    # 4. Hapus simbol dan angka
    text = re.sub(r'[^\w\s]', ' ', text) # Ganti simbol dengan spasi agar kata tidak menempel
    text = re.sub(r'\d+', '', text)

    # 5. Tokenisasi & Stopwords (The missing piece)
    tokens = word_tokenize(text)
    cleaned_tokens = [w for w in tokens if w not in stop_words and len(w) > 2]

    return " ".join(cleaned_tokens)

In [7]:
df_preprocessed = df['full_text_translated'].fillna('').apply(preprocess_tweet_v2)
df_preprocessed = pd.concat([df_preprocessed, df['image_url']], axis=1)

In [8]:
df_preprocessed = df_preprocessed[df_preprocessed['full_text_translated'].apply(lambda x: len(x.split()) >= 4)]
df_preprocessed.count()

full_text_translated    19049
image_url                6000
dtype: int64

In [9]:
docs = df_preprocessed['full_text_translated'].tolist()
images  = df_preprocessed['image_url'].tolist()
for i in range(len(images)):
    if pd.isna(images[i]):
        images[i] = None
random.shuffle(docs)
docs[:10]

['use cursor windsurf vibe crypto coding ariel johna',
 'claude code cursor vibecoding boring forms business insight building use even features idea vibecode also vibe business let figure code use cases epics',
 'vibecoding using tab tab tab could wrong',
 'even vibecoding requires coding',
 'vibecoding onchain games live tomorrow book calendars get popcorn vibecoding game actually pretty easy',
 'vibecoding going become like drug addiction give tokens tokens please money need tokens right',
 'excited see partner bring hosting infrastructure vibe coders code dead long live vibecoding',
 'tuesday may come vibecode secure agents onchain superpowers',
 'vibecoding claude via starlink moving vehicle cool cool',
 'chat true say vibecoding gon viral let know comments make sure like subscribe follow tips tricks nothing case cursor windsurf old dot new lovable know']

In [11]:
# Top2Vec topic modeling
documents = [doc for doc in docs if isinstance(doc, str) and doc.strip()]
tokenized_documents = [doc.split() for doc in documents]

dictionary = Dictionary(tokenized_documents)
dictionary.filter_extremes(no_below=2, no_above=0.5)

print(f"tokenized documents: {tokenized_documents[:5]}")
print(f"Number of documents: {len(documents)}")
print(f"Number of unique tokens: {len(dictionary)}")



tokenized documents: [['use', 'cursor', 'windsurf', 'vibe', 'crypto', 'coding', 'ariel', 'johna'], ['claude', 'code', 'cursor', 'vibecoding', 'boring', 'forms', 'business', 'insight', 'building', 'use', 'even', 'features', 'idea', 'vibecode', 'also', 'vibe', 'business', 'let', 'figure', 'code', 'use', 'cases', 'epics'], ['vibecoding', 'using', 'tab', 'tab', 'tab', 'could', 'wrong'], ['even', 'vibecoding', 'requires', 'coding'], ['vibecoding', 'onchain', 'games', 'live', 'tomorrow', 'book', 'calendars', 'get', 'popcorn', 'vibecoding', 'game', 'actually', 'pretty', 'easy']]
Number of documents: 19049
Number of unique tokens: 10517


In [12]:
print(dictionary.token2id)

{'ariel': 0, 'coding': 1, 'crypto': 2, 'cursor': 3, 'johna': 4, 'use': 5, 'vibe': 6, 'windsurf': 7, 'also': 8, 'boring': 9, 'building': 10, 'business': 11, 'cases': 12, 'claude': 13, 'code': 14, 'even': 15, 'features': 16, 'figure': 17, 'forms': 18, 'idea': 19, 'insight': 20, 'let': 21, 'vibecode': 22, 'could': 23, 'tab': 24, 'using': 25, 'wrong': 26, 'requires': 27, 'actually': 28, 'book': 29, 'calendars': 30, 'easy': 31, 'game': 32, 'games': 33, 'get': 34, 'live': 35, 'onchain': 36, 'pretty': 37, 'tomorrow': 38, 'addiction': 39, 'become': 40, 'drug': 41, 'give': 42, 'going': 43, 'like': 44, 'money': 45, 'need': 46, 'please': 47, 'right': 48, 'tokens': 49, 'bring': 50, 'coders': 51, 'dead': 52, 'excited': 53, 'hosting': 54, 'infrastructure': 55, 'long': 56, 'partner': 57, 'see': 58, 'agents': 59, 'come': 60, 'may': 61, 'secure': 62, 'superpowers': 63, 'tuesday': 64, 'cool': 65, 'moving': 66, 'starlink': 67, 'vehicle': 68, 'via': 69, 'case': 70, 'chat': 71, 'comments': 72, 'dot': 73, '

In [13]:
import top2vec
print(top2vec.__version__)

1.0.36


In [16]:
top2vec_model = Top2Vec(
    documents=documents,
    contextual_top2vec=True,
    ngram_vocab=True,
    min_count=10,
    embedding_model='all-MiniLM-L6-v2',    
    speed='learn',
    workers=2,
    keep_documents=True,
    verbose=True,
 )

topic_words, topic_word_scores, topic_nums = top2vec_model.get_topics()



2026-04-12 20:49:34,508 - top2vec - INFO - Pre-processing documents for training
2026-04-12 20:49:35,434 - top2vec - INFO - Creating vocabulary embedding
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5280.30it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Embedding vocabulary: 100%|██████████| 25/25 [00:03<00:00,  8.29it/s]
2026-04-12 20:49:42,033 - top2vec - INFO - Create contextualized document embeddings
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6334.51it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored whe

In [17]:
topic_sizes, topic_ids = top2vec_model.get_topic_sizes()
pd.DataFrame({
    "topic_id": topic_ids,
    "size": topic_sizes
}).sort_values("size", ascending=False).head(10)

,topic_id,size
0,1,406020
1,0,5475


In [18]:
print(len(topic_sizes))

2


In [19]:
# Coherence metrics for Top2Vec topics (C_V and NPMI)
topic_pairs = []
for topic_num, topic in zip(topic_nums, topic_words):
    cleaned_topic_terms = []
    for term in topic[:10]:
        # ngram terms like "vibe coding" are split so they can be matched to dictionary tokens
        for tok in str(term).split():
            if tok in dictionary.token2id:
                cleaned_topic_terms.append(tok)
    # keep token order, drop duplicates
    cleaned_topic_terms = list(dict.fromkeys(cleaned_topic_terms))
    if len(cleaned_topic_terms) >= 2:
        topic_pairs.append((topic_num, cleaned_topic_terms))

topics_for_coherence = [words for _, words in topic_pairs]
topic_nums_for_coherence = [topic_num for topic_num, _ in topic_pairs]

if not topics_for_coherence:
    raise ValueError("No valid topics left for coherence calculation. Try lowering dictionary filtering or increasing topic words.")

coherence_cv_model = CoherenceModel(
    topics=topics_for_coherence,
    texts=tokenized_documents,
    dictionary=dictionary,
    coherence='c_v',
)

coherence_npmi_model = CoherenceModel(
    topics=topics_for_coherence,
    texts=tokenized_documents,
    dictionary=dictionary,
    coherence='c_npmi',
)

coherence_cv = coherence_cv_model.get_coherence()
coherence_npmi = coherence_npmi_model.get_coherence()

print(f"Total topics used for coherence: {len(topics_for_coherence)}")
print(f"C_V coherence: {coherence_cv:.4f}")
print(f"NPMI coherence: {coherence_npmi:.4f}")

topic_coherence = pd.DataFrame({
    'topic_num': topic_nums_for_coherence,
    'topic_words': [' '.join(words) for words in topics_for_coherence],
    'c_v': coherence_cv_model.get_coherence_per_topic(),
    'npmi': coherence_npmi_model.get_coherence_per_topic(),
})

topic_coherence.sort_values('c_v', ascending=False).head(10)

Total topics used for coherence: 2
C_V coherence: 0.2501
NPMI coherence: -0.1255


,topic_num,topic_words,c_v,npmi
1,1,vibe debugging coding marketing coded designin...,0.297761,-0.158421
0,0,music visuals dance club consumption vibe desi...,0.202525,-0.092545


In [20]:
# get all topic words values
topic_words_values = topic_coherence.sort_values('c_v', ascending=False).head(10)['topic_words'].values
print(topic_words_values)


['vibe debugging coding marketing coded designing vibecoded projects investing vibecode cleanup camp dont'
 'music visuals dance club consumption vibe designing art marketing coding vibecoded projects coded investing']


In [ ]:
# Save model
top2vec_model.save("top2vec_contextual_ngram_c10.model")

# Load model (di masa depan)
# from top2vec import Top2Vec
top2vec_model = Top2Vec.load("top2vec_contextual_ngram_c10.model")



# Load di kemudian hari
# embeddings = np.load("top2vec_contextual_ngram_c10_embeddings.npy")


# Simpan Dictionary
dictionary.save("top2vec_contextual_ngram_c10.dict")

# Simpan tokenized_documents (menggunakan pickle karena list of lists)
import pickle
with open("top2vec_contextual_ngram_c10_tokenized_docs.pkl", "wb") as f:
    pickle.dump(tokenized_documents, f)
    
    
    
# Simpan ringkasan topik dan metriknya
topic_coherence.to_csv("top2vec_contextual_ngram_c10_metrics_summary.csv", index=False)

# Simpan representasi topik (10 kata teratas)
top_words_df = pd.DataFrame(topic_words)
top_words_df.to_csv("top2vec_contextual_ngram_c10_topic_words.csv", index=False)